# Chapter 16 — Statistical Process Control (SPC)
*Track 4: Engineers*

## 🎯 Learning Objectives
- Build Shewhart control charts (X̄, R, p charts)
- Detect out-of-control signals using Western Electric rules
- Distinguish common-cause from special-cause variation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Control Charts

A **control chart** monitors a process statistic over time.
Control limits are set at ±3σ of the statistic (not of the data).

**X̄ chart (mean chart):**
- UCL = $\bar{\bar X} + 3\frac{\hat\sigma}{\sqrt n}$
- LCL = $\bar{\bar X} - 3\frac{\hat\sigma}{\sqrt n}$

The **false alarm rate** with ±3σ limits ≈ 0.27% per point (assuming normality).

In [ ]:
# Simulate a manufacturing process
n_subgroups = 30
subgroup_size = 5
true_mean = 50.0
true_std = 2.0

# First 20 groups in-control, last 10 with shift
data = np.vstack([
    rng.normal(true_mean, true_std, (20, subgroup_size)),
    rng.normal(true_mean + 2.5, true_std, (10, subgroup_size)),
])
xbar = data.mean(axis=1)
R    = data.max(axis=1) - data.min(axis=1)

# Control limits (using first 20 in-control groups)
xbar_bar = xbar[:20].mean()
R_bar = R[:20].mean()
A2 = 0.577  # for subgroup size 5
D3, D4 = 0, 2.114

UCL_x = xbar_bar + A2 * R_bar
LCL_x = xbar_bar - A2 * R_bar

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
ax = axes[0]
ax.plot(range(1, n_subgroups+1), xbar, "b-o", markersize=4)
ax.axhline(xbar_bar, color="green", linewidth=1.5, label="CL")
ax.axhline(UCL_x, color="red", linewidth=1.5, linestyle="--", label="UCL")
ax.axhline(LCL_x, color="red", linewidth=1.5, linestyle="--", label="LCL")
out_of_control = np.where((xbar > UCL_x) | (xbar < LCL_x))[0]
ax.scatter(out_of_control+1, xbar[out_of_control], color="red", zorder=5, s=80, label="OOC")
ax.set_title("X̄ Chart"); ax.legend(fontsize=9)
ax.set_xlabel("Subgroup"); ax.set_ylabel("X̄")

ax = axes[1]
UCL_r = D4 * R_bar
LCL_r = D3 * R_bar
ax.plot(range(1, n_subgroups+1), R, "b-o", markersize=4)
ax.axhline(R_bar, color="green", linewidth=1.5, label="CL")
ax.axhline(UCL_r, color="red", linewidth=1.5, linestyle="--", label="UCL")
ax.axhline(LCL_r, color="red", linewidth=1.5, linestyle="--", label="LCL")
ax.set_title("R Chart"); ax.legend(fontsize=9)
ax.set_xlabel("Subgroup"); ax.set_ylabel("Range R")
plt.tight_layout(); plt.show()
print(f"Out-of-control subgroups (X̄ chart): {out_of_control + 1}")

## 2. Math Walkthrough — Process Capability

**Cp** measures potential capability (spread only):
$$C_p = \frac{USL - LSL}{6\hat\sigma}$$

**Cpk** measures actual capability (includes centering):
$$C_{pk} = \min\left(\frac{USL - \bar x}{3\hat\sigma},\; \frac{\bar x - LSL}{3\hat\sigma}\right)$$

Rule of thumb: $C_{pk} \geq 1.33$ for a capable process.

In [ ]:
# Process capability analysis
USL, LSL = 56, 44
process_data = rng.normal(50.5, 1.8, 500)  # slightly off-center

sigma_hat = process_data.std(ddof=1)
xbar_est  = process_data.mean()
Cp  = (USL - LSL) / (6 * sigma_hat)
Cpk = min((USL - xbar_est) / (3*sigma_hat), (xbar_est - LSL) / (3*sigma_hat))
ppm_defects = (stats.norm.sf(USL, xbar_est, sigma_hat) +
               stats.norm.cdf(LSL, xbar_est, sigma_hat)) * 1e6

print(f"Process mean: {xbar_est:.2f}, σ: {sigma_hat:.2f}")
print(f"Cp={Cp:.3f}, Cpk={Cpk:.3f}")
print(f"Estimated defects: {ppm_defects:.0f} ppm")
print("Capable ✅" if Cpk >= 1.33 else "NOT capable ❌")

x_range = np.linspace(40, 60, 300)
plt.figure(figsize=(9, 4))
plt.hist(process_data, bins=40, density=True, alpha=0.5, label="Process data")
plt.plot(x_range, stats.norm.pdf(x_range, xbar_est, sigma_hat), "b-", lw=2)
plt.axvline(USL, color="red", lw=2, linestyle="--", label=f"USL={USL}")
plt.axvline(LSL, color="red", lw=2, linestyle="--", label=f"LSL={LSL}")
plt.fill_between(x_range[x_range>USL], stats.norm.pdf(x_range[x_range>USL], xbar_est, sigma_hat), alpha=0.4, color="red", label="Defects")
plt.fill_between(x_range[x_range<LSL], stats.norm.pdf(x_range[x_range<LSL], xbar_est, sigma_hat), alpha=0.4, color="red")
plt.title(f"Process Capability: Cp={Cp:.2f}, Cpk={Cpk:.2f}")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# p-chart for attribute data (proportion defective)
n_inspected = 100
p_true_in  = 0.03  # first 20 batches
p_true_out = 0.08  # last 10 batches
defects = np.concatenate([rng.binomial(n_inspected, p_true_in, 20),
                           rng.binomial(n_inspected, p_true_out, 10)])
p_i = defects / n_inspected
pbar = p_i[:20].mean()
UCL_p = pbar + 3*np.sqrt(pbar*(1-pbar)/n_inspected)
LCL_p = max(0, pbar - 3*np.sqrt(pbar*(1-pbar)/n_inspected))

plt.figure(figsize=(12, 4))
plt.plot(range(1, 31), p_i, "b-o", markersize=4)
plt.axhline(pbar, color="green", lw=1.5, label=f"p̄={pbar:.3f}")
plt.axhline(UCL_p, color="red", lw=1.5, linestyle="--", label=f"UCL={UCL_p:.3f}")
plt.axhline(LCL_p, color="red", lw=1.5, linestyle="--", label=f"LCL={LCL_p:.3f}")
ooc = np.where((p_i > UCL_p) | (p_i < LCL_p))[0]
plt.scatter(ooc+1, p_i[ooc], color="red", s=80, zorder=5)
plt.title("p-Chart — Proportion Defective"); plt.legend()
plt.xlabel("Batch"); plt.ylabel("p"); plt.tight_layout(); plt.show()